### This is the Custom CNN model which will train on the Dataset Provided and save the trained weights.
NOTE: All the necessary files are provided in the google drive link which can be accessed from Github repo in Readme file

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from torchvision.models import densenet121, DenseNet121_Weights
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import (roc_auc_score, precision_recall_fscore_support,
                             roc_curve, auc, precision_recall_curve)
import matplotlib.pyplot as plt

# ====== Configuration ======
# All the necessaey files are provided in the google drive link which can be accessed from Github repo in Readme file
LABEL_PATH = './train_data.csv'
IMAGE_FOLDER = '/path/to/images'
MODEL_SAVE_PATH = './finetuned_weights.pth'
PLOT_SAVE_DIR = './plots'
NUM_LABELS = 15
IMAGE_SIZE = 224
NUM_EPOCHS = 10
BATCH_SIZE = 8
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs(PLOT_SAVE_DIR, exist_ok=True)

# ====== Dataset ======
class ChestXrayDataset(Dataset):
    def __init__(self, csv_path, image_folder, transform=None):
        self.df = pd.read_csv(csv_path)
        self.image_folder = image_folder
        self.transform = transform

        all_labels = set()
        for labels in self.df['Finding Labels']:
            all_labels.update(labels.split('|'))
        self.labels = sorted(list(all_labels))
        self.label_map = {label: i for i, label in enumerate(self.labels)}
        self.df['encoded'] = self.df['Finding Labels'].apply(self.encode_labels)

    def encode_labels(self, label_str):
        vec = np.zeros(len(self.label_map), dtype=np.float32)
        for label in label_str.split('|'):
            if label in self.label_map:
                vec[self.label_map[label]] = 1.0
        return vec

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_folder, row['Image Index'])
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(row['encoded'], dtype=torch.float32)
        return img, label

# ====== Focal Loss ======
class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        inputs = torch.sigmoid(inputs)
        BCE = F.binary_cross_entropy(inputs, targets, reduction='none')
        pt = torch.where(targets == 1, inputs, 1 - inputs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        loss = alpha_t * (1 - pt) ** self.gamma * BCE
        return loss.mean()

# ====== Model ======
class CheXNetFineTune(nn.Module):
    def __init__(self, num_labels):
        super().__init__()
        base = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_labels)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.classifier(x)

# ====== Threshold Tuning ======
def threshold_tuning(y_true, y_scores):
    thresholds = []
    for i in range(y_true.shape[1]):
        best_thresh = 0.5
        best_f1 = 0
        for t in np.arange(0.1, 0.9, 0.01):
            preds = (y_scores[:, i] >= t).astype(int)
            _, _, f1, _ = precision_recall_fscore_support(y_true[:, i], preds, average='binary', zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thresh = t
        thresholds.append(best_thresh)
    return np.array(thresholds)

# ====== Evaluation ======
def evaluate(model, dataloader, class_names):
    model.eval()
    all_labels, all_scores = [], []
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating"):
            outputs = torch.sigmoid(model(images.to(DEVICE)))
            all_scores.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())

    y_true = np.vstack(all_labels)
    y_scores = np.vstack(all_scores)
    thresholds = threshold_tuning(y_true, y_scores)
    y_pred = (y_scores >= thresholds).astype(int)

    auc_macro = roc_auc_score(y_true, y_scores, average='macro')
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro', zero_division=0)

    per_auc = [roc_auc_score(y_true[:, i], y_scores[:, i]) for i in range(y_true.shape[1])]
    per_p, per_r, per_f1, _ = precision_recall_fscore_support(y_true, y_pred, average=None, zero_division=0)

    print(f"Macro AUC: {auc_macro:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    metrics_df = pd.DataFrame({
        'Label': class_names,
        'AUC': per_auc,
        'Precision': per_p,
        'Recall': per_r,
        'F1 Score': per_f1
    })
    metrics_df.to_csv(os.path.join(PLOT_SAVE_DIR, 'evaluation_metrics.csv'), index=False)

    return y_true, y_scores, class_names

# ====== Plotting ======
def plot_roc_curves(y_true, y_scores, class_names):
    plt.figure(figsize=(12, 9))
    for i in range(y_true.shape[1]):
        fpr, tpr, _ = roc_curve(y_true[:, i], y_scores[:, i])
        plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC={auc(fpr, tpr):.2f})')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel('FPR')
    plt.ylabel('TPR')
    plt.title('ROC Curves')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_SAVE_DIR, 'roc_curves.png'))

def plot_pr_curves(y_true, y_scores, class_names):
    plt.figure(figsize=(12, 9))
    for i in range(y_true.shape[1]):
        precision, recall, _ = precision_recall_curve(y_true[:, i], y_scores[:, i])
        plt.plot(recall, precision, label=class_names[i])
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title('Precision-Recall Curves')
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(PLOT_SAVE_DIR, 'precision_recall_curves.png'))

# ====== Main ======
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
dataset = ChestXrayDataset(LABEL_PATH, IMAGE_FOLDER, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True)
class_names = dataset.labels

model = CheXNetFineTune(num_labels=NUM_LABELS).to(DEVICE)

label_matrix = np.stack(dataset.df['encoded'].to_numpy())
alpha = 1.0 - (label_matrix.sum(axis=0) / len(dataset))
alpha_tensor = torch.tensor(alpha, dtype=torch.float32).to(DEVICE)
criterion = FocalLoss(alpha_tensor)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0
    for images, labels in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}: Loss = {total_loss / len(dataloader):.4f}")

torch.save(model.state_dict(), MODEL_SAVE_PATH)
print(f"Model saved to {MODEL_SAVE_PATH}")

y_true, y_scores, _ = evaluate(model, dataloader, class_names)
plot_roc_curves(y_true, y_scores, class_names)
plot_pr_curves(y_true, y_scores, class_names)
